# Test Evaluation with LLM Ground Truth

This notebook is a small end-to-end evaluation smoke test. It uses the 20-query LLM test ground truth file and evaluates the saved model prediction files in `method_runs_top200/`.

This is not the human post-checking step. The goal is to verify that the retrieval run files can be evaluated directly against LLM relevance labels.


## 1. Setup and Paths

The prediction files are the frozen outputs of each retrieval method. The test ground truth file contains 50 LLM relevance labels per query. The default evaluation mode is `raw_topk_unjudged_as_zero`, which evaluates the original top-K ranking returned by each method. Documents outside the judged pool are treated as non-relevant for metric computation.


In [1]:
from __future__ import annotations

import csv
import json
import math
from collections import Counter, defaultdict
from pathlib import Path
from typing import Optional


def find_finalproject_root(start_path: Optional[Path] = None) -> Path:
    """Find the Finalproject directory by walking upward from the current path."""
    current_path = (start_path or Path.cwd()).resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if candidate_path.name == "Finalproject":
            return candidate_path
        nested_finalproject = candidate_path / "Finalproject"
        if nested_finalproject.exists() and nested_finalproject.is_dir():
            return nested_finalproject.resolve()
    raise FileNotFoundError("Could not find the Finalproject directory.")


FINALPROJECT_ROOT = find_finalproject_root()
NOTEBOOK_OUTPUT_DIR = FINALPROJECT_ROOT / "notebooks" / "rec_and_eval" / "groundtruth_outputs"
ANNOTATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "annotation"
METHOD_RUNS_DIR = NOTEBOOK_OUTPUT_DIR / "method_runs_top200"
METRICS_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "metrics"

TEST_LABELS_PATH = ANNOTATION_OUTPUT_DIR / "llm_groundtruth_labels.jsonl"
TEST_METRICS_OUTPUT_PATH = METRICS_OUTPUT_DIR / "test_eval_metrics_raw_topk_from_llm_groundtruth.csv"
TEST_PER_QUERY_OUTPUT_PATH = METRICS_OUTPUT_DIR / "test_eval_per_query_metrics_raw_topk_from_llm_groundtruth.csv"

EVALUATION_MODE = "raw_topk_unjudged_as_zero"
DEFAULT_CUTOFFS = [10, 20, 50]
RELEVANCE_THRESHOLD = 1

print("Finalproject root:", FINALPROJECT_ROOT)
print("Test labels path:", TEST_LABELS_PATH)
print("Method runs dir:", METHOD_RUNS_DIR)
print("Evaluation mode:", EVALUATION_MODE)
print("Cutoffs:", DEFAULT_CUTOFFS)


Finalproject root: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject
Test labels path: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\annotation\llm_groundtruth_labels.jsonl
Method runs dir: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\method_runs_top200
Evaluation mode: raw_topk_unjudged_as_zero
Cutoffs: [10, 20, 50]


## 2. Load Test Ground Truth and Prediction Runs

The LLM label file is loaded as graded qrels: `query_id -> doc_id -> relevance`. Each method run file is loaded as a ranked list of predicted `doc_id` values. The run files are not recomputed here.


In [2]:
def load_jsonl_records(jsonl_path: Path) -> list[dict]:
    """Load a JSONL file into dictionaries."""
    records = []
    with jsonl_path.open("r", encoding="utf-8") as input_file:
        for line_number, line in enumerate(input_file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON at {jsonl_path}:{line_number}") from error
    return records


def load_llm_qrels(labels_path: Path) -> dict[int, dict[int, int]]:
    """Load flattened LLM labels as query_id -> doc_id -> graded relevance."""
    label_records = load_jsonl_records(labels_path)
    qrels: dict[int, dict[int, int]] = defaultdict(dict)

    for record in label_records:
        query_id = int(record["query_id"])
        doc_id = int(record["doc_id"])
        relevance = int(record["relevance"])
        if relevance not in {0, 1, 2, 3}:
            raise ValueError(f"Invalid relevance={relevance} for query_id={query_id}, doc_id={doc_id}")
        if doc_id in qrels[query_id]:
            raise ValueError(f"Duplicate qrel for query_id={query_id}, doc_id={doc_id}")
        qrels[query_id][doc_id] = relevance

    return dict(qrels)


def load_method_run(run_path: Path) -> tuple[str, dict[int, list[int]]]:
    """Load one method JSONL run file as query_id -> ranked doc_id list."""
    run_records = load_jsonl_records(run_path)
    method_names = {str(record.get("method_name", run_path.stem.replace("_top200", ""))) for record in run_records}
    method_name = sorted(method_names)[0] if method_names else run_path.stem.replace("_top200", "")
    if len(method_names) > 1:
        print(f"Warning: multiple method_name values in {run_path.name}: {sorted(method_names)}")

    run_by_query: dict[int, list[int]] = {}
    for record in run_records:
        query_id = int(record["query_id"])
        retrieved_docs = sorted(record["retrieved_docs"], key=lambda item: int(item["rank"]))
        ranked_doc_ids = [int(item["doc_id"]) for item in retrieved_docs]
        if len(ranked_doc_ids) != len(set(ranked_doc_ids)):
            raise ValueError(f"Duplicate doc_id in run {run_path.name}, query_id={query_id}")
        run_by_query[query_id] = ranked_doc_ids

    return method_name, run_by_query


def load_all_method_runs(method_runs_dir: Path) -> dict[str, dict[int, list[int]]]:
    """Load every top-200 method run JSONL file."""
    runs = {}
    for run_path in sorted(method_runs_dir.glob("*_top200.jsonl")):
        method_name, run_by_query = load_method_run(run_path)
        runs[method_name] = run_by_query
    if not runs:
        raise FileNotFoundError(f"No *_top200.jsonl files found in {method_runs_dir}")
    return runs


qrels = load_llm_qrels(TEST_LABELS_PATH)
method_runs = load_all_method_runs(METHOD_RUNS_DIR)

labels_per_query = Counter(len(doc_labels) for doc_labels in qrels.values())
print("Number of qrel queries:", len(qrels))
print("Judged labels per query distribution:", dict(labels_per_query))
print("Loaded method runs:", sorted(method_runs.keys()))
print("Queries per method:", {method_name: len(run) for method_name, run in method_runs.items()})


Number of qrel queries: 364
Judged labels per query distribution: {50: 364}
Loaded method runs: ['Hybrid_Content', 'Hybrid_TFIDF_SBERT', 'Ingredient_TFIDF', 'Keyword', 'RARec_Late_Fusion', 'SBERT_FAISS', 'TFIDF']
Queries per method: {'Hybrid_Content': 500, 'Hybrid_TFIDF_SBERT': 500, 'Ingredient_TFIDF': 500, 'Keyword': 500, 'RARec_Late_Fusion': 500, 'SBERT_FAISS': 500, 'TFIDF': 500}


## 3. Metric Functions

The qrels are graded from 0 to 3. `nDCG@K` uses the graded labels directly. Precision, recall, MRR, and MAP use a binary relevance threshold where any label greater than or equal to `1` is relevant.


In [3]:
def relevance_for_doc(qrels_for_query: dict[int, int], doc_id: int) -> int:
    """Return the judged relevance for a document, or zero if it is not judged."""
    return int(qrels_for_query.get(int(doc_id), 0))


def dcg_at_k(relevance_values: list[int], cutoff: int) -> float:
    """Compute graded DCG@K."""
    return sum(
        ((2 ** relevance - 1) / math.log2(rank_index + 2))
        for rank_index, relevance in enumerate(relevance_values[:cutoff])
    )


def ndcg_at_k(predicted_doc_ids: list[int], qrels_for_query: dict[int, int], cutoff: int) -> float:
    """Compute graded nDCG@K for one query."""
    predicted_relevance = [relevance_for_doc(qrels_for_query, doc_id) for doc_id in predicted_doc_ids[:cutoff]]
    ideal_relevance = sorted(qrels_for_query.values(), reverse=True)[:cutoff]
    ideal_dcg = dcg_at_k(ideal_relevance, cutoff)
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(predicted_relevance, cutoff) / ideal_dcg


def precision_at_k(predicted_doc_ids: list[int], qrels_for_query: dict[int, int], cutoff: int) -> float:
    """Compute binary Precision@K with a fixed K denominator."""
    top_k_doc_ids = predicted_doc_ids[:cutoff]
    relevant_hits = sum(
        1
        for doc_id in top_k_doc_ids
        if relevance_for_doc(qrels_for_query, doc_id) >= RELEVANCE_THRESHOLD
    )
    return relevant_hits / cutoff


def recall_at_k(predicted_doc_ids: list[int], qrels_for_query: dict[int, int], cutoff: int) -> float:
    """Compute binary Recall@K over all relevant judged documents."""
    total_relevant = sum(1 for relevance in qrels_for_query.values() if relevance >= RELEVANCE_THRESHOLD)
    if total_relevant == 0:
        return 0.0
    relevant_hits = sum(
        1
        for doc_id in predicted_doc_ids[:cutoff]
        if relevance_for_doc(qrels_for_query, doc_id) >= RELEVANCE_THRESHOLD
    )
    return relevant_hits / total_relevant


def reciprocal_rank_at_k(predicted_doc_ids: list[int], qrels_for_query: dict[int, int], cutoff: int) -> float:
    """Compute binary reciprocal rank at K."""
    for rank_index, doc_id in enumerate(predicted_doc_ids[:cutoff], start=1):
        if relevance_for_doc(qrels_for_query, doc_id) >= RELEVANCE_THRESHOLD:
            return 1.0 / rank_index
    return 0.0


def average_precision_at_k(predicted_doc_ids: list[int], qrels_for_query: dict[int, int], cutoff: int) -> float:
    """Compute binary AP@K normalized by min(number of judged relevant docs, K)."""
    total_relevant = sum(1 for relevance in qrels_for_query.values() if relevance >= RELEVANCE_THRESHOLD)
    denominator = min(total_relevant, cutoff)
    if denominator == 0:
        return 0.0

    relevant_hits = 0
    precision_sum = 0.0
    for rank_index, doc_id in enumerate(predicted_doc_ids[:cutoff], start=1):
        if relevance_for_doc(qrels_for_query, doc_id) >= RELEVANCE_THRESHOLD:
            relevant_hits += 1
            precision_sum += relevant_hits / rank_index
    return precision_sum / denominator


## 4. Evaluate Saved Prediction Runs

In `raw_topk_unjudged_as_zero` mode, each model keeps its original ranking. For every cutoff, the notebook takes the first K predicted documents exactly as the model returned them. If a predicted document is not present in the LLM-judged pool, its relevance is treated as 0. This is the main user-facing evaluation mode because it does not alter the model ranking.


In [4]:
def prepare_ranked_docs_for_evaluation(
    predicted_doc_ids: list[int],
    qrels_for_query: dict[int, int],
    evaluation_mode: str,
) -> list[int]:
    """Prepare one query ranking according to the evaluation mode."""
    if evaluation_mode == "raw_topk_unjudged_as_zero":
        return list(predicted_doc_ids)
    if evaluation_mode == "judged_only":
        judged_doc_ids = set(qrels_for_query.keys())
        return [doc_id for doc_id in predicted_doc_ids if doc_id in judged_doc_ids]
    raise ValueError(f"Unknown evaluation_mode: {evaluation_mode}")


def evaluate_method_run(
    method_name: str,
    run_by_query: dict[int, list[int]],
    qrels: dict[int, dict[int, int]],
    cutoffs: list[int],
    evaluation_mode: str,
) -> tuple[list[dict], list[dict]]:
    """Evaluate one method and return aggregate rows plus per-query rows."""
    per_query_rows = []

    for query_id, qrels_for_query in sorted(qrels.items()):
        raw_predicted_doc_ids = run_by_query.get(query_id, [])
        evaluated_doc_ids = prepare_ranked_docs_for_evaluation(
            predicted_doc_ids=raw_predicted_doc_ids,
            qrels_for_query=qrels_for_query,
            evaluation_mode=evaluation_mode,
        )

        for cutoff in cutoffs:
            per_query_rows.append(
                {
                    "method_name": method_name,
                    "query_id": query_id,
                    "cutoff": cutoff,
                    "evaluation_mode": evaluation_mode,
                    "judged_docs_for_query": len(qrels_for_query),
                    "raw_prediction_depth": len(raw_predicted_doc_ids),
                    "evaluated_prediction_depth": len(evaluated_doc_ids),
                    "unjudged_docs_at_cutoff": sum(
                        1 for doc_id in evaluated_doc_ids[:cutoff] if doc_id not in qrels_for_query
                    ),
                    "precision": precision_at_k(evaluated_doc_ids, qrels_for_query, cutoff),
                    "recall": recall_at_k(evaluated_doc_ids, qrels_for_query, cutoff),
                    "mrr": reciprocal_rank_at_k(evaluated_doc_ids, qrels_for_query, cutoff),
                    "ndcg": ndcg_at_k(evaluated_doc_ids, qrels_for_query, cutoff),
                    "map": average_precision_at_k(evaluated_doc_ids, qrels_for_query, cutoff),
                }
            )

    aggregate_rows = []
    for cutoff in cutoffs:
        rows_at_cutoff = [row for row in per_query_rows if row["cutoff"] == cutoff]
        aggregate_rows.append(
            {
                "method_name": method_name,
                "cutoff": cutoff,
                "evaluation_mode": evaluation_mode,
                "num_queries": len(rows_at_cutoff),
                "mean_evaluated_prediction_depth": sum(row["evaluated_prediction_depth"] for row in rows_at_cutoff)
                / len(rows_at_cutoff),
                "mean_unjudged_docs_at_cutoff": sum(row["unjudged_docs_at_cutoff"] for row in rows_at_cutoff)
                / len(rows_at_cutoff),
                "precision": sum(row["precision"] for row in rows_at_cutoff) / len(rows_at_cutoff),
                "recall": sum(row["recall"] for row in rows_at_cutoff) / len(rows_at_cutoff),
                "mrr": sum(row["mrr"] for row in rows_at_cutoff) / len(rows_at_cutoff),
                "ndcg": sum(row["ndcg"] for row in rows_at_cutoff) / len(rows_at_cutoff),
                "map": sum(row["map"] for row in rows_at_cutoff) / len(rows_at_cutoff),
            }
        )

    return aggregate_rows, per_query_rows


all_metric_rows = []
all_per_query_rows = []
for method_name, run_by_query in sorted(method_runs.items()):
    aggregate_rows, per_query_rows = evaluate_method_run(
        method_name=method_name,
        run_by_query=run_by_query,
        qrels=qrels,
        cutoffs=DEFAULT_CUTOFFS,
        evaluation_mode=EVALUATION_MODE,
    )
    all_metric_rows.extend(aggregate_rows)
    all_per_query_rows.extend(per_query_rows)


def print_metric_table(metric_rows: list[dict], cutoff: int) -> None:
    """Print a compact text table for one cutoff."""
    rows = [row for row in metric_rows if row["cutoff"] == cutoff]
    rows = sorted(rows, key=lambda row: row["ndcg"], reverse=True)
    print(f"\nMetrics @{cutoff} sorted by nDCG")
    print(
        f"{'method_name':28s} {'depth':>7s} {'unjudged':>9s} {'P':>8s} {'R':>8s} {'MRR':>8s} {'nDCG':>8s} {'MAP':>8s}"
    )
    for row in rows:
        print(
            f"{row['method_name'][:28]:28s} "
            f"{row['mean_evaluated_prediction_depth']:7.2f} "
            f"{row['mean_unjudged_docs_at_cutoff']:9.2f} "
            f"{row['precision']:8.4f} "
            f"{row['recall']:8.4f} "
            f"{row['mrr']:8.4f} "
            f"{row['ndcg']:8.4f} "
            f"{row['map']:8.4f}"
        )


for cutoff in DEFAULT_CUTOFFS:
    print_metric_table(all_metric_rows, cutoff)



Metrics @10 sorted by nDCG
method_name                    depth  unjudged        P        R      MRR     nDCG      MAP
Hybrid_TFIDF_SBERT            200.00      0.05   0.6429   0.3588   0.9214   0.7106   0.6483
TFIDF                         200.00      0.86   0.5764   0.3210   0.9036   0.6602   0.5698
Keyword                       200.00      5.49   0.2953   0.2094   0.9604   0.5982   0.2991
SBERT_FAISS                   200.00      2.76   0.3854   0.2096   0.7192   0.4325   0.3399
Hybrid_Content                200.00      0.88   0.4626   0.2350   0.6838   0.4228   0.3937
RARec_Late_Fusion             200.00      3.42   0.3687   0.1943   0.6759   0.3531   0.3175
Ingredient_TFIDF              200.00      3.79   0.2659   0.1163   0.4355   0.1904   0.1919

Metrics @20 sorted by nDCG
method_name                    depth  unjudged        P        R      MRR     nDCG      MAP
Hybrid_TFIDF_SBERT            200.00      0.60   0.5712   0.5610   0.9215   0.7341   0.6149
TFIDF                   

## 5. Save Test Evaluation Artifacts

The aggregate CSV is useful for quick comparison across methods. The per-query CSV is useful for later error analysis.


In [5]:
def write_csv_rows(output_path: Path, rows: list[dict]) -> None:
    """Write a list of dictionaries to CSV."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        raise ValueError(f"No rows to write: {output_path}")
    fieldnames = list(rows[0].keys())
    with output_path.open("w", encoding="utf-8-sig", newline="") as output_file:
        writer = csv.DictWriter(output_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


write_csv_rows(TEST_METRICS_OUTPUT_PATH, all_metric_rows)
write_csv_rows(TEST_PER_QUERY_OUTPUT_PATH, all_per_query_rows)

print("Saved aggregate metrics to:", TEST_METRICS_OUTPUT_PATH)
print("Saved per-query metrics to:", TEST_PER_QUERY_OUTPUT_PATH)


Saved aggregate metrics to: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\metrics\test_eval_metrics_raw_topk_from_llm_groundtruth.csv
Saved per-query metrics to: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\metrics\test_eval_per_query_metrics_raw_topk_from_llm_groundtruth.csv
